In [ ]:
pip install xgboost catboost shap scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
import pickle
import hashlib
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.preprocessing import (
    LabelEncoder, StandardScaler)
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve
)
import xgboost  as xgb
import catboost as cb
import shap

# ============================================================
# CONFIGURATION
# ============================================================
PATH_DATA   = "../06_Model_Data/"
PATH_OUTPUT = "../07_Model_Output/"

for sub in ["plots", "shap", "scores",
            "models", "cache"]:
    os.makedirs(
        os.path.join(PATH_OUTPUT, sub),
        exist_ok=True)

PATH_CACHE   = os.path.join(PATH_OUTPUT, "cache")
RANDOM_STATE = 42
TOP_DECILE   = 0.10
SPW_GRID     = [10, 20, 47]

HIGH_THRESH = 0.90
MED_THRESH  = 0.80

PRIORITY_1_N = 500
PRIORITY_2_N = 2000

# ── NEW JOINER TRIAGE GUARDRAIL ───────────────────────────
# Aligned to ABT threshold of 18m.
# Employees with tenure < 18m will have
# their priority tier capped at P3-Monitor.
# This is POST-MODEL triage logic, not a model change.
NEW_JOINER_TRIAGE_MONTHS = 18

C_NAVY  = "#1F3864"
C_BLUE  = "#2E75B6"
C_AMBER = "#F0A500"
C_RED   = "#C00000"
C_GREEN = "#1A5C2A"
C_LGRAY = "#D9D9D9"

ID_COLS    = ["Employee_ID", "Month_Date", "Year"]
EXTRA_DROP = [
    "Termination_Date",
    "Termination_Date_dt",
    "Contingent Worker Type",
    "Worker Type",
    "Performance_Score",
    "_perf_this_year",
    "_perf_prior_year",
    # FIX: "promo_fallback_used" removed from drop list.
    # It was renamed to "has_promo_history" in NB02 and
    # is now kept as a model feature so trees can split:
    # (mslp high AND has_promo_history=1) = genuinely overdue
    # (mslp high AND has_promo_history=0) = imputed, less signal
]

CAT_COLS = [
    "AAP Group", "City", "Company",
    "Cost Center",
    "Disability Status",
    "Doctorate Degree Name",
    "EEO Category", "Employee Type",
    "Job Family Group",
    "Job Profile Summary",
    "Level 2 Org", "Level 3 Org",
    "Level 4 Org", "Level 5 Org",
    "Level 6 Org", "Level 7 Org",
    "Level 8 Org", "Level 9 Org",
    "Location Code",
    "On Cycle Promotion - To Management Level",
    "Primary Position Applicable "
    "Exemption Status",
    "Professional Group Investment "
    "Professionals Flag",
    "Professional Group Support Staff Flag",
    "Professional Group Technologist Flag",
    "RFP - Professional Category",
    "Region", "State",
    "Supervisory Organization",
    "US Asian+ Flag", "US Black+ Flag",
    "US Ethnicity Short Name",
    "US Latinx+ Flag", "US Other+ Flag",
    "US White+ Flag", "Veteran Status",
    "Gender",
    "country_grouped",
    "job_family_grouped",
]

PATH_PARQUET = (
    "../05_Master_Data/"
    "Master_Headcount_Full.parquet")


# ============================================================
# CACHE HELPERS
# ============================================================
def get_data_hash():
    train_path = os.path.join(
        PATH_DATA, "abt_train.parquet")
    live_path  = os.path.join(
        PATH_DATA, "abt_live.parquet")
    s_tr = os.stat(train_path)
    s_li = os.stat(live_path)
    h = (f"{s_tr.st_mtime}_{s_tr.st_size}_"
         f"{s_li.st_mtime}_{s_li.st_size}")
    return hashlib.md5(
        h.encode()).hexdigest()[:10]

def cache_exists(name):
    return os.path.exists(
        os.path.join(
            PATH_CACHE, f"{name}.pkl"))

def save_cache(name, obj):
    path = os.path.join(
        PATH_CACHE, f"{name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"  Cache saved: {name}.pkl")

def load_cache(name):
    path = os.path.join(
        PATH_CACHE, f"{name}.pkl")
    with open(path, "rb") as f:
        obj = pickle.load(f)
    print(f"  Cache loaded: {name}.pkl")
    return obj

def print_cache_status():
    import datetime
    print("\n  Current cache contents:")
    files = [
        f for f in os.listdir(PATH_CACHE)
        if f.endswith(".pkl")]
    if not files:
        print("    (empty)")
        return
    for cf in sorted(files):
        fp = os.path.join(PATH_CACHE, cf)
        sz = os.path.getsize(fp)/1024/1024
        dt = datetime.datetime.fromtimestamp(
            os.path.getmtime(fp)
        ).strftime("%Y-%m-%d %H:%M")
        print(f"    {cf:<45} "
              f"{sz:>6.1f} MB  {dt}")


# ============================================================
# RISK BAND ASSIGNMENT
# ============================================================
def assign_risk_bands(prob_array):
    return np.where(
        prob_array >= HIGH_THRESH, "High",
        np.where(
            prob_array >= MED_THRESH,
            "Medium", "Low"
        )
    )


# ============================================================
# PRIORITY TIER ASSIGNMENT + NEW JOINER GUARDRAIL
# ============================================================
def assign_priority_tiers(
        df_scored,
        tenure_col="tenure_months",
        apply_nj_guardrail=True):
    """
    Assign priority tiers within High+Medium band.

    NEW JOINER GUARDRAIL:
      Employees with tenure < NEW_JOINER_TRIAGE_MONTHS
      are capped at P3-Monitor regardless of score.
      Threshold aligned to ABT: 18 months.

      This is NOT a model change.
      It is triage logic — fully auditable.
    """
    df = df_scored.copy()
    df = df.sort_values(
        "attrition_score", ascending=False
    ).reset_index(drop=True)

    df["priority_tier"] = "P4 - Watch List"

    hm_mask = df["risk_band"].isin(
        ["High", "Medium"])
    hm_idx  = df[hm_mask].index

    p1_end = min(PRIORITY_1_N, len(hm_idx))
    p2_end = min(
        PRIORITY_1_N + PRIORITY_2_N,
        len(hm_idx))

    df.loc[hm_idx[:p1_end],
           "priority_tier"] = "P1 - Urgent"
    df.loc[hm_idx[p1_end:p2_end],
           "priority_tier"] = "P2 - High"
    df.loc[hm_idx[p2_end:],
           "priority_tier"] = "P3 - Monitor"

    # ── NEW JOINER GUARDRAIL ──────────────────
    if apply_nj_guardrail and \
            tenure_col in df.columns:
        nj_mask = (
            df[tenure_col] <
            NEW_JOINER_TRIAGE_MONTHS
        )
        cap_mask = (
            nj_mask &
            df["priority_tier"].isin(
                ["P1 - Urgent",
                 "P2 - High"])
        )
        n_capped = int(cap_mask.sum())
        if n_capped > 0:
            df.loc[cap_mask,
                   "priority_tier"] = \
                "P3 - Monitor"
            print(
                f"\n  NEW JOINER GUARDRAIL:")
            print(
                f"    {n_capped:,} employees"
                f" capped P1/P2 -> P3")
            print(
                f"    Tenure < "
                f"{NEW_JOINER_TRIAGE_MONTHS}m"
                f" (aligned to ABT threshold)")
            print(
                f"    Triage logic only —"
                f" not a model change.")
            if "Region" in df.columns:
                capped_df = df[cap_mask]
                print(
                    f"\n    Capped by Region:")
                print(
                    capped_df["Region"]
                    .value_counts()
                    .to_string())
            if "mgmt_ordinal" in df.columns:
                print(
                    f"\n    Capped by Level:")
                print(
                    capped_df["mgmt_ordinal"]
                    .value_counts()
                    .sort_index()
                    .to_string())
        else:
            print(
                f"\n  NEW JOINER GUARDRAIL:"
                f" 0 employees to cap"
                f" (tenure >= "
                f"{NEW_JOINER_TRIAGE_MONTHS}m"
                f" for all H+M)")
    elif apply_nj_guardrail and \
            tenure_col not in df.columns:
        print(
            f"  WARNING: '{tenure_col}'"
            f" not found — guardrail skipped")

    return df


# ============================================================
# SHAP RECOMPUTE DECISION
# ============================================================
print("=" * 65)
print("SHAP FULL-POPULATION CACHE CHECK")
print("=" * 65)

SHAP_FULL_CACHE  = "shap_full_population"
shap_cache_found = cache_exists(
    SHAP_FULL_CACHE)

if shap_cache_found:
    print(
        f"\n  Full-population SHAP"
        f" cache found.")
    print_cache_status()
    print(
        "\n  Recompute SHAP for all"
        " live employees?")
    print(
        "  (~4 min on latest snapshot)"
        " Type 'yes' to recompute: ")
    try:
        user_input = input().strip().lower()
    except EOFError:
        user_input = "no"
    RECOMPUTE_SHAP = (user_input == "yes")
    print(
        f"  -> "
        f"{'Recomputing' if RECOMPUTE_SHAP else 'Using cache'}")
else:
    print(
        "\n  No SHAP cache found —"
        " will compute after training.")
    RECOMPUTE_SHAP = True


# ============================================================
# SECTION 1 — LOAD & VALIDATE SPLITS
# ============================================================
print("\n" + "=" * 65)
print("SECTION 1 — LOAD & VALIDATE SPLITS")
print("=" * 65)

df_train = pd.read_parquet(
    os.path.join(
        PATH_DATA, "abt_train.parquet"))
df_val   = pd.read_parquet(
    os.path.join(
        PATH_DATA, "abt_val.parquet"))
df_test  = pd.read_parquet(
    os.path.join(
        PATH_DATA, "abt_test.parquet"))
df_live  = pd.read_parquet(
    os.path.join(
        PATH_DATA, "abt_live.parquet"))

for df in [df_train, df_val,
           df_test, df_live]:
    df["Month_Date"] = pd.to_datetime(
        df["Month_Date"])

# ── ABT fix validation ────────────────────
print("\n  ABT fix validation checks:")
for nm, sp in [
    ("TRAIN", df_train),
    ("VAL",   df_val),
    ("TEST",  df_test),
    ("LIVE",  df_live)
]:
    if "promotion_gap_months" in sp.columns:
        n_neg = int(
            (sp["promotion_gap_months"]
             < 0).sum())
        nj_ct = int(
            sp["is_new_joiner"].sum()
        ) if "is_new_joiner" in sp.columns \
            else -1
        # FIX: check has_promo_history exists
        hph_ct = int(
            sp["has_promo_history"].sum()
        ) if "has_promo_history" in sp.columns \
            else -1
        hph_pct = (
            hph_ct / max(len(sp), 1) * 100
        ) if hph_ct >= 0 else 0.0
        n_nan_mslp = int(
            sp["months_since_last_promotion"].isna().sum()
        ) if "months_since_last_promotion" in sp.columns \
            else 0
        status = (
            "OK" if n_neg == 0 else "FAIL")
        print(
            f"    {nm:<6} | neg gaps: {n_neg:>5} [{status}]"
            f" | NJ: {nj_ct:>6,}"
            f" | has_promo: {hph_ct:>6,} ({hph_pct:.1f}%)"
            f" | mslp NaN: {n_nan_mslp:>6,}")
        assert n_neg == 0, (
            f"FATAL: negative promotion"
            f" gaps in {nm} -- re-run ABT!")
        if hph_ct == -1:
            print(f"    WARNING: has_promo_history "
                  f"missing from {nm} -- re-run ABT02!")
print("  All ABT fix assertions passed")

print(
    f"\n  {'Split':<6} | {'Rows':>9} | "
    f"{'Employees':>10} | "
    f"{'Leavers':>8} | "
    f"{'Pos Rate':>10} | "
    f"{'New Joiners':>12}")
print("  " + "-" * 72)
for name, split in [
    ("TRAIN", df_train),
    ("VAL",   df_val),
    ("TEST",  df_test),
    ("LIVE",  df_live)
]:
    n   = len(split)
    emp = split["Employee_ID"].nunique()
    nj  = int(
        split["is_new_joiner"].sum()
    ) if "is_new_joiner" in split.columns \
        else 0
    if "attrition_label" in split.columns:
        lvo = int(
            split["attrition_label"].sum())
        pct = float(
            split["attrition_label"].mean()
            * 100)
        print(
            f"  {name:<6} | {n:>9,} |"
            f" {emp:>10,} |"
            f" {lvo:>8,} | {pct:>9.3f}% |"
            f" {nj:>12,}")
    else:
        print(
            f"  {name:<6} | {n:>9,} |"
            f" {emp:>10,} |"
            f" {'N/A':>8} | {'N/A':>10} |"
            f" {nj:>12,}")

n_pos = int(
    df_train["attrition_label"].sum())
n_neg_cls = int(
    (df_train["attrition_label"] == 0)
    .sum())
print(
    f"\n  Class ratio (TRAIN):"
    f" 1:{n_neg_cls // max(n_pos, 1)}"
    f" ({n_neg_cls:,} stayers :"
    f" {n_pos:,} leavers)")


# ============================================================
# SECTION 2 — PREPROCESSING
# ============================================================
print("\n" + "=" * 65)
print("SECTION 2 — PREPROCESSING")
print("=" * 65)

def preprocess(df_tr, df_va,
               df_te, df_li):

    # ── STEP 0: Filter live to latest
    # snapshot month ONLY ─────────────────
    # abt_live has multiple rows per employee
    # (one per rolling snapshot month).
    # We want SHAP and scoring to reflect the
    # employee's CURRENT state (Feb 2026),
    # not a historical snapshot.
    # This also reduces X_live from ~99k
    # to ~25k rows — 4x faster SHAP.
    latest_live_month = df_li[
        "Month_Date"].max()
    n_before = len(df_li)
    df_li = df_li[
        df_li["Month_Date"] ==
        latest_live_month
    ].copy()
    n_after = len(df_li)
    print(
        f"  Live snapshot filter:"
        f" {latest_live_month.strftime('%b %Y')}"
        f" | {n_before:,} rows ->"
        f" {n_after:,} rows"
        f" ({n_before - n_after:,} removed)")
    print(
        f"  Unique live employees:"
        f" {df_li['Employee_ID'].nunique():,}")

    # Check: should be 1 row per employee
    dupes = (
        df_li["Employee_ID"]
        .duplicated().sum())
    if dupes > 0:
        print(
            f"  WARNING: {dupes:,} duplicate"
            f" Employee_IDs in live after"
            f" month filter — keeping last")
        df_li = (
            df_li
            .sort_values("Month_Date")
            .groupby(
                "Employee_ID",
                as_index=False)
            .last()
        )
    else:
        print(
            f"  1 row per employee"
            f" confirmed ✅")

    # live_ids for score output
    live_ids = df_li[[
        "Employee_ID",
        "Month_Date"]].copy()

    # Preserve tenure for guardrail
    live_tenure = None
    if "tenure_months" in df_li.columns:
        live_tenure = df_li[[
            "Employee_ID",
            "tenure_months"]].copy()

    # ── STEP 1: Drop ID and extra cols ────
    drop_cols = ID_COLS + EXTRA_DROP
    for df in [df_tr, df_va,
               df_te, df_li]:
        df.drop(
            columns=[
                c for c in drop_cols
                if c in df.columns],
            inplace=True)

    y_train = df_tr.pop(
        "attrition_label"
    ).astype(int).values
    y_val   = df_va.pop(
        "attrition_label"
    ).astype(int).values
    y_test  = df_te.pop(
        "attrition_label"
    ).astype(int).values
    if "attrition_label" in df_li.columns:
        df_li.drop(
            columns=["attrition_label"],
            inplace=True)

    # ── STEP 2: Encode categoricals ───────
    print(
        "  Encoding categoricals"
        " (fit on train)...")
    cat_present    = [
        c for c in CAT_COLS
        if c in df_tr.columns]
    label_encoders = {}
    for col in cat_present:
        le = LabelEncoder()
        train_vals = (
            df_tr[col].astype(str)
            .fillna("Missing")
            .unique().tolist()
        )
        le.fit(train_vals + ["Unknown"])
        label_encoders[col] = le
        for df in [df_tr, df_va,
                   df_te, df_li]:
            vals = (
                df[col].astype(str)
                .fillna("Missing"))
            vals = vals.apply(
                lambda x: x
                if x in le.classes_
                else "Unknown")
            df[col] = le.transform(vals)

    # ── STEP 3: Impute numerics ───────────
    # FIX: mslp and promotion_gap_months now
    # arrive as float32 with NaN for employees
    # who have no promo history in the CSV
    # (non-new-joiners only). SimpleImputer fills
    # these NaN values with the median of employees
    # who DO have promo history. This breaks the
    # previous mslp = tenure_months identity for
    # the fallback population.
    print(
        "  Imputing numerics"
        " (median, fit on train)...")
    num_cols = (
        df_tr
        .select_dtypes(include=[np.number])
        .columns.tolist()
    )

    # Diagnostic: show NaN rates before imputation
    mslp_nan_tr = df_tr["months_since_last_promotion"].isna().sum()         if "months_since_last_promotion" in df_tr.columns else 0
    gap_nan_tr  = df_tr["promotion_gap_months"].isna().sum()         if "promotion_gap_months" in df_tr.columns else 0
    print(f"  Pre-impute NaN in mslp (TRAIN) : {mslp_nan_tr:,}")
    print(f"  Pre-impute NaN in gap  (TRAIN) : {gap_nan_tr:,}")

    imputer = SimpleImputer(
        strategy="median")
    df_tr[num_cols] = (
        imputer.fit_transform(
            df_tr[num_cols]))
    for df in [df_va, df_te, df_li]:
        nc = [
            c for c in num_cols
            if c in df.columns]
        df[nc] = imputer.transform(df[nc])

    # Show what the imputer filled mslp NaN with
    if "months_since_last_promotion" in num_cols:
        mslp_idx = num_cols.index(
            "months_since_last_promotion")
        imputed_mslp = imputer.statistics_[mslp_idx]
        print(f"  mslp imputed value (median of history employees): "
              f"{imputed_mslp:.1f}m")

    feature_names = df_tr.columns.tolist()
    print(
        f"  Features:"
        f" {len(feature_names)} total"
        f" ({len(cat_present)} categorical,"
        f" {len(num_cols)} numeric)")

    return (
        df_tr.values.astype(np.float32),
        y_train,
        df_va.values.astype(np.float32),
        y_val,
        df_te.values.astype(np.float32),
        y_test,
        df_li.values.astype(np.float32),
        live_ids,
        feature_names,
        label_encoders,
        imputer,
        live_tenure
    )


(X_train, y_train,
 X_val,   y_val,
 X_test,  y_test,
 X_live,  live_ids,
 feature_names,
 label_encoders,
 imputer,
 live_tenure) = preprocess(
    df_train.copy(), df_val.copy(),
    df_test.copy(),  df_live.copy()
)
gc.collect()

print(
    f"  X_train: {X_train.shape} |"
    f" X_val: {X_val.shape} |"
    f" X_test: {X_test.shape} |"
    f" X_live: {X_live.shape}")
print(
    f"  NOTE: X_live is latest"
    f" snapshot only (1 row/employee)")

save_cache("label_encoders", label_encoders)
save_cache("imputer",        imputer)
save_cache("feature_names",  feature_names)


# ============================================================
# SECTION 3 — EVALUATION HELPER
# ============================================================
def evaluate(name, y_true, y_prob,
             top_decile=TOP_DECILE,
             verbose=True):
    auc_roc = roc_auc_score(
        y_true, y_prob)
    auc_pr  = average_precision_score(
        y_true, y_prob)
    n_top   = max(
        1, int(len(y_prob) * top_decile))
    top_idx = np.argsort(
        y_prob)[::-1][:n_top]
    recall_decile = (
        y_true[top_idx].sum() /
        max(y_true.sum(), 1))
    prec, rec, thresh = (
        precision_recall_curve(
            y_true, y_prob))
    f1 = (2 * prec * rec /
          np.maximum(prec + rec, 1e-9))
    best_i      = np.argmax(f1)
    best_thresh = (
        float(thresh[best_i])
        if best_i < len(thresh)
        else 0.5)
    best_f1 = float(f1[best_i])
    if verbose:
        print(f"  [{name}]")
        print(
            f"    AUC-ROC   :"
            f" {auc_roc:.4f}")
        print(
            f"    AUC-PR    :"
            f" {auc_pr:.4f}")
        print(
            f"    Recall@"
            f"{top_decile*100:.0f}%:"
            f" {recall_decile*100:.1f}%")
        print(
            f"    Best F1   :"
            f" {best_f1:.4f}"
            f" @ {best_thresh:.3f}")
    return dict(
        name=name,
        auc_roc=auc_roc,
        auc_pr=auc_pr,
        recall_decile=recall_decile,
        best_f1=best_f1,
        best_thresh=best_thresh,
        prec=prec, rec=rec,
        fpr=roc_curve(y_true, y_prob)[0],
        tpr=roc_curve(y_true, y_prob)[1]
    )


# ============================================================
# SECTION 4 — BASELINE: LOGISTIC REGRESSION
# ============================================================
print("\n" + "=" * 65)
print("SECTION 4 — BASELINE: LR")
print("=" * 65)

scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_va_sc = scaler.transform(X_val)
X_te_sc = scaler.transform(X_test)

lr = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_jobs=-1)
lr.fit(X_tr_sc, y_train)

lr_val_prob  = (
    lr.predict_proba(X_va_sc)[:, 1])
lr_test_prob = (
    lr.predict_proba(X_te_sc)[:, 1])

print("\n  Validation:")
lr_val_m  = evaluate(
    "LR VAL",  y_val,  lr_val_prob)
print("  Test:")
lr_test_m = evaluate(
    "LR TEST", y_test, lr_test_prob)

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5))
axes[0].plot(
    lr_test_m["fpr"], lr_test_m["tpr"],
    color=C_NAVY, lw=2,
    label=f"AUC-ROC="
          f"{lr_test_m['auc_roc']:.4f}")
axes[0].plot([0,1],[0,1],"k--",lw=1)
axes[0].set(
    xlabel="FPR", ylabel="TPR",
    title="LR ROC (Test)")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(
    lr_test_m["rec"], lr_test_m["prec"],
    color=C_BLUE, lw=2,
    label=f"AUC-PR="
          f"{lr_test_m['auc_pr']:.4f}")
axes[1].set(
    xlabel="Recall",
    ylabel="Precision",
    title="LR PR (Test)")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "plots",
        "lr_roc_pr.png"),
    dpi=150, bbox_inches="tight")
plt.close()
print("  Plot saved: lr_roc_pr.png")


# ============================================================
# SECTION 5 — CHAMPION: XGBOOST
# ============================================================
print("\n" + "=" * 65)
print("SECTION 5 — CHAMPION: XGBOOST")
print("=" * 65)

best_spw        = None
best_auc_pr_xgb = -1
best_xgb        = None

for spw in SPW_GRID:
    print(
        f"\n  Training XGBoost"
        f" spw={spw}...")
    model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        scale_pos_weight=spw,
        eval_metric="aucpr",
        early_stopping_rounds=30,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False)
    val_prob = (
        model.predict_proba(X_val)[:, 1])
    m = evaluate(
        f"XGB spw={spw} VAL",
        y_val, val_prob)
    if m["auc_pr"] > best_auc_pr_xgb:
        best_auc_pr_xgb = m["auc_pr"]
        best_spw        = spw
        best_xgb        = model

print(
    f"\n  Best spw={best_spw}"
    f" AUC-PR={best_auc_pr_xgb:.4f}")

xgb_val_prob  = (
    best_xgb.predict_proba(X_val)[:, 1])
xgb_test_prob = (
    best_xgb.predict_proba(X_test)[:, 1])
xgb_live_prob = (
    best_xgb.predict_proba(X_live)[:, 1])

print("\n  Test (XGBoost):")
xgb_test_m = evaluate(
    "XGB TEST", y_test, xgb_test_prob)

evals = best_xgb.evals_result()
aucpr = evals.get(
    "validation_0", {}
).get("aucpr", [])
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    aucpr, color=C_NAVY, lw=2,
    label="Val AUC-PR")
ax.axvline(
    best_xgb.best_iteration,
    color=C_RED, lw=1.5,
    linestyle="--",
    label=f"Best:"
          f" {best_xgb.best_iteration}")
ax.set(
    xlabel="Round",
    ylabel="AUC-PR",
    title=f"XGB Learning Curve"
          f" spw={best_spw}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "plots",
        "xgb_learning_curve.png"),
    dpi=150, bbox_inches="tight")
plt.close()

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5))
axes[0].plot(
    xgb_test_m["fpr"],
    xgb_test_m["tpr"],
    color=C_NAVY, lw=2,
    label=f"AUC-ROC="
          f"{xgb_test_m['auc_roc']:.4f}")
axes[0].plot([0,1],[0,1],"k--",lw=1)
axes[0].set(
    xlabel="FPR", ylabel="TPR",
    title="XGB ROC (Test)")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(
    xgb_test_m["rec"],
    xgb_test_m["prec"],
    color=C_BLUE, lw=2,
    label=f"AUC-PR="
          f"{xgb_test_m['auc_pr']:.4f}")
axes[1].set(
    xlabel="Recall",
    ylabel="Precision",
    title="XGB PR (Test)")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "plots",
        "xgb_roc_pr.png"),
    dpi=150, bbox_inches="tight")
plt.close()
print("  Plots saved: xgb_*.png")

with open(os.path.join(
        PATH_OUTPUT, "models",
        "xgb_champion.pkl"), "wb") as f:
    pickle.dump(best_xgb, f)
print("  Model saved: xgb_champion.pkl")


# ============================================================
# SECTION 6 — CHALLENGER: CATBOOST
# ============================================================
print("\n" + "=" * 65)
print("SECTION 6 — CHALLENGER: CATBOOST")
print("=" * 65)

cb_model = cb.CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    scale_pos_weight=best_spw,
    eval_metric="PRAUC",
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    verbose=0,
)
cb_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=None)

cb_val_prob  = (
    cb_model.predict_proba(X_val)[:, 1])
cb_test_prob = (
    cb_model.predict_proba(X_test)[:, 1])
cb_live_prob = (
    cb_model.predict_proba(X_live)[:, 1])

print("  Validation:")
cb_val_m  = evaluate(
    "CatBoost VAL",
    y_val,  cb_val_prob)
print("  Test:")
cb_test_m = evaluate(
    "CatBoost TEST",
    y_test, cb_test_prob)

cb_model.save_model(os.path.join(
    PATH_OUTPUT, "models",
    "catboost_challenger.cbm"))
print(
    "  Model saved:"
    " catboost_challenger.cbm")


# ============================================================
# SECTION 7 — MODEL COMPARISON
# ============================================================
print("\n" + "=" * 65)
print("SECTION 7 — MODEL COMPARISON")
print("=" * 65)

all_test_m = [
    lr_test_m, xgb_test_m, cb_test_m]
print(
    f"\n  {'Model':<22}"
    f" {'AUC-ROC':>8}"
    f" {'AUC-PR':>8}"
    f" {'R@10%':>7}"
    f" {'F1':>7}"
    f" {'Thresh':>8}")
print("  " + "-" * 66)
for m in all_test_m:
    print(
        f"  {m['name']:<22}"
        f" {m['auc_roc']:>8.4f}"
        f" {m['auc_pr']:>8.4f}"
        f" {m['recall_decile']*100:>6.1f}%"
        f" {m['best_f1']:>7.4f}"
        f" {m['best_thresh']:>8.3f}")

champion_m = max(
    all_test_m,
    key=lambda x: x["auc_pr"])
print(f"\n  Champion: {champion_m['name']}")

if "XGB" in champion_m["name"]:
    champ_test_prob = xgb_test_prob
    champ_live_prob = xgb_live_prob
    champ_model     = best_xgb
elif "CatBoost" in champion_m["name"]:
    champ_test_prob = cb_test_prob
    champ_live_prob = cb_live_prob
    champ_model     = cb_model
else:
    champ_test_prob = lr_test_prob
    champ_live_prob = lr.predict_proba(
        scaler.transform(X_live))[:, 1]
    champ_model     = lr

fig, axes = plt.subplots(
    1, 2, figsize=(16, 6))
for m, col, lbl in zip(
        all_test_m,
        [C_NAVY, C_BLUE, C_AMBER],
        ["LR", "XGBoost", "CatBoost"]):
    axes[0].plot(
        m["fpr"], m["tpr"],
        color=col, lw=2,
        label=f"{lbl}"
              f" ({m['auc_roc']:.4f})")
    axes[1].plot(
        m["rec"], m["prec"],
        color=col, lw=2,
        label=f"{lbl}"
              f" ({m['auc_pr']:.4f})")
axes[0].plot([0,1],[0,1],"k--",lw=1)
axes[0].set(
    xlabel="FPR", ylabel="TPR",
    title="ROC — All Models (Test)")
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)
axes[1].set(
    xlabel="Recall",
    ylabel="Precision",
    title="PR — All Models (Test)")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "plots",
        "model_comparison_roc_pr.png"),
    dpi=150, bbox_inches="tight")
plt.close()
print(
    "  Plot saved:"
    " model_comparison_roc_pr.png")


# ============================================================
# SECTION 8 — RISK BAND SCORING
# ============================================================
print("\n" + "=" * 65)
print("SECTION 8 — RISK BAND SCORING")
print("=" * 65)

print(f"\n  Thresholds (absolute):")
print(f"    High   >= {HIGH_THRESH}")
print(f"    Medium >= {MED_THRESH}")
print(f"    Low    <  {MED_THRESH}")
print(f"\n  New joiner triage guardrail:")
print(
    f"    P1/P2 -> P3 for tenure"
    f" < {NEW_JOINER_TRIAGE_MONTHS}m"
    f" (aligned to ABT threshold)")

# TEST scores
test_meta = pd.read_parquet(
    os.path.join(
        PATH_DATA, "abt_test.parquet"),
    columns=[
        "Employee_ID", "Month_Date",
        "tenure_months"])
test_meta["attrition_score"] = (
    champ_test_prob)
test_meta["risk_band"] = assign_risk_bands(
    champ_test_prob)
test_meta["actual_leaver"] = y_test

# New joiner check on TEST
nj_test = test_meta[
    test_meta["tenure_months"] <
    NEW_JOINER_TRIAGE_MONTHS]
nj_high = nj_test[
    nj_test["risk_band"] == "High"]
nj_prec = float(
    nj_high["actual_leaver"].sum() /
    max(len(nj_high), 1) * 100)
print(f"\n  New joiner check (TEST set):")
print(
    f"    Total new joiners : "
    f"{len(nj_test):,}")
print(
    f"    Scored High       : "
    f"{len(nj_high):,}")
print(
    f"    Actual leavers    : "
    f"{int(nj_high['actual_leaver'].sum()):,}")
print(
    f"    Precision (nj-Hi) : "
    f"{nj_prec:.1f}%")
if nj_prec < 30:
    print(
        f"    NOTE: Low precision"
        f" confirms guardrail justified.")
else:
    print(
        f"    NOTE: Reasonable precision"
        f" — genuine signal present.")

print(f"\n  TEST risk band distribution:")
for band in ["High", "Medium", "Low"]:
    bd   = test_meta[
        test_meta["risk_band"] == band]
    n    = len(bd)
    nlv  = int(bd["actual_leaver"].sum())
    pct  = float(
        nlv / max(y_test.sum(), 1) * 100)
    prec = float(nlv / max(n, 1) * 100)
    print(
        f"    {band:<8}: {n:>7,} emps |"
        f" {nlv:>5,} leavers |"
        f" {pct:.1f}% captured |"
        f" Prec {prec:.1f}%")

# LIVE scores — already 1 row/employee
# from preprocess latest-month filter
live_ids["attrition_score"] = champ_live_prob
live_ids["risk_band"] = assign_risk_bands(
    champ_live_prob)

# No dedup needed — already 1 row/employee
live_scores_dedup = live_ids.copy()

# Merge tenure for guardrail
if live_tenure is not None:
    live_scores_dedup = (
        live_scores_dedup.merge(
            live_tenure
            .drop_duplicates("Employee_ID"),
            on="Employee_ID",
            how="left"))
    print(
        f"\n  tenure_months merged"
        f" for guardrail ✅")

print(
    f"\n  LIVE scores"
    f" (latest snapshot only,"
    f" 1 row/employee):"
    f" {len(live_scores_dedup):,}"
    f" employees")
for band in ["High", "Medium", "Low"]:
    n = int(
        (live_scores_dedup["risk_band"]
         == band).sum())
    pct = float(
        n / max(
            len(live_scores_dedup), 1)
        * 100)
    print(
        f"    {band:<8}: {n:>6,}"
        f" ({pct:.1f}%)")

# Save raw scores
test_meta.to_parquet(
    os.path.join(
        PATH_OUTPUT, "scores",
        "test_scores.parquet"),
    index=False)
test_meta.to_csv(
    os.path.join(
        PATH_OUTPUT, "scores",
        "test_scores.csv"),
    index=False)
live_scores_dedup.to_parquet(
    os.path.join(
        PATH_OUTPUT, "scores",
        "live_scores.parquet"),
    index=False)
live_scores_dedup.to_csv(
    os.path.join(
        PATH_OUTPUT, "scores",
        "live_scores.csv"),
    index=False)
print(
    "\n  Scores saved:"
    " test_scores & live_scores")

# Score distribution plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(
    champ_test_prob, bins=100,
    color=C_LGRAY,
    edgecolor="white", linewidth=0.3)
ax.axvspan(
    MED_THRESH, HIGH_THRESH,
    alpha=0.25, color=C_AMBER,
    label="Medium risk")
ax.axvspan(
    HIGH_THRESH, 1.0,
    alpha=0.35, color=C_RED,
    label="High risk")
ax.axvline(
    HIGH_THRESH, color=C_RED,
    lw=1.5, linestyle="--",
    label=f"High>={HIGH_THRESH}")
ax.axvline(
    MED_THRESH, color=C_AMBER,
    lw=1.5, linestyle="--",
    label=f"Med>={MED_THRESH}")
ax.set(
    xlabel="Attrition Score",
    ylabel="Employees",
    title="Score Distribution —"
          " Absolute Thresholds")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "plots",
        "score_distribution.png"),
    dpi=150, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(10, 5))
bl, precs, recs = [], [], []
for band in ["High", "Medium", "Low"]:
    bd  = test_meta[
        test_meta["risk_band"] == band]
    nlv = int(bd["actual_leaver"].sum())
    bl.append(band)
    precs.append(
        float(nlv / max(len(bd), 1) * 100))
    recs.append(
        float(
            nlv /
            max(y_test.sum(), 1) * 100))
x = np.arange(len(bl))
w = 0.35
b1 = ax.bar(
    x - w/2, precs, w,
    color=C_NAVY, label="Precision %")
b2 = ax.bar(
    x + w/2, recs, w,
    color=C_BLUE,
    label="% Leavers Captured")
ax.set(
    xticks=x, xticklabels=bl,
    ylabel="%",
    title="Precision & Capture"
          " by Risk Band")
ax.legend()
ax.grid(alpha=0.3, axis="y")
for bar in list(b1) + list(b2):
    ax.text(
        bar.get_x() +
        bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{bar.get_height():.1f}%",
        ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "plots",
        "risk_band_precision_recall.png"),
    dpi=150, bbox_inches="tight")
plt.close()
print(
    "  Plots saved: score_distribution,"
    " risk_band_precision_recall")


# ============================================================
# SECTION 8B — ENRICH LIVE SCORES
# ============================================================
print("\n" + "=" * 65)
print("SECTION 8B — ENRICHING LIVE SCORES")
print("=" * 65)

# Active HC filter
print(
    "  Building active HC filter"
    " from raw parquet...")
try:
    _feb = pd.read_parquet(
        PATH_PARQUET,
        columns=[
            "Employee_ID", "Month_Date",
            "HC_CTT_EE"])
    _feb["Month_Date"] = pd.to_datetime(
        _feb["Month_Date"])
    _feb["Employee_ID"] = (
        _feb["Employee_ID"]
        .astype(str).str.strip())

    latest_snap = _feb["Month_Date"].max()
    print(
        f"  Latest snapshot : "
        f"{latest_snap.strftime('%b %Y')}")

    active_ids = set(
        _feb[
            (_feb["Month_Date"] ==
             latest_snap) &
            (_feb["HC_CTT_EE"] == 1)
        ]["Employee_ID"].unique()
    )
    del _feb
    gc.collect()

    print(
        f"  Active (CTT=1)   :"
        f" {len(active_ids):,}")
    before_filter = len(live_scores_dedup)
    live_active = live_scores_dedup[
        live_scores_dedup["Employee_ID"]
        .isin(active_ids)
    ].copy()
    after_filter = len(live_active)
    print(
        f"  Before filter :"
        f" {before_filter:,}")
    print(
        f"  After filter  :"
        f" {after_filter:,}")
    print(
        f"  Removed       :"
        f" {before_filter - after_filter:,}"
        f" (already left)")

except Exception as e:
    print(
        f"  Could not load parquet: {e}")
    print(f"  Proceeding without filter")
    live_active = live_scores_dedup.copy()

# Load abt_live features
# Filter to latest month only
# (matches what preprocess did for X_live)
ENRICH_COLS = [
    "Employee_ID", "Month_Date",
    "Region", "mgmt_ordinal",
    "tenure_months",
    "time_in_title_months",
    "Is_Acquisition_Hire",
    "is_rehire",
    "EOS_Score", "EOS_Score_Delta",
    "Months_Since_Survey",
    "months_since_last_promotion",
    "manager_changes_24m",
    "current_manager_tenure_months",
    "promotion_gap_months",
    "eos_and_perf_both_declined",
    "is_early_tenure",
    "is_new_joiner",
    "is_honeymoon_cliff",
    "is_long_tenure",
    "tenure_stability_ratio",
    "early_tenure_band",
    "snapshot_month",
    "snapshot_year",
    "has_promo_history",
]

abt_live_path = os.path.join(
    PATH_DATA, "abt_live.parquet")

if os.path.exists(abt_live_path):
    abt_raw = pd.read_parquet(abt_live_path)
    abt_raw["Month_Date"] = pd.to_datetime(
        abt_raw["Month_Date"])

    enrich_avail = [
        c for c in ENRICH_COLS
        if c in abt_raw.columns]

    # Filter to latest month only
    # to match X_live snapshot
    latest_abt = abt_raw["Month_Date"].max()
    abt_latest = abt_raw[
        abt_raw["Month_Date"] ==
        latest_abt
    ][enrich_avail].copy()

    print(
        f"\n  abt_live snapshot :"
        f" {latest_abt.strftime('%b %Y')}")
    print(
        f"  Rows (latest month only) :"
        f" {len(abt_latest):,}")
    print(
        f"  Employees :"
        f" {abt_latest['Employee_ID'].nunique():,}")

    # Validate no negative gaps
    if "promotion_gap_months" in \
            abt_latest.columns:
        n_neg = int(
            (abt_latest[
                 "promotion_gap_months"]
             < 0).sum())
        print(
            f"  Negative gaps :"
            f" {n_neg}  <- must be 0")
        assert n_neg == 0, (
            "FATAL: negative gaps in"
            " abt_live — re-run ABT!")
        print(
            f"  Negative gap assert :"
            f" PASSED ✅")

    if "is_new_joiner" in abt_latest.columns:
        nj_live = int(
            abt_latest[
                "is_new_joiner"].sum())
        print(
            f"  New joiners in live :"
            f" {nj_live:,}")

    # Drop Month_Date before merge
    # to avoid _x/_y suffixes
    abt_for_merge = abt_latest.drop(
        columns=["Month_Date"],
        errors="ignore")

    # Merge features onto active scores
    live_enriched = live_active.merge(
        abt_for_merge,
        on="Employee_ID",
        how="left"
    )

    # Resolve tenure duplicate if exists
    # (safety net — should not occur after
    # latest-month filter alignment)
    if "tenure_months_x" in \
            live_enriched.columns and \
            "tenure_months_y" in \
            live_enriched.columns:
        live_enriched["tenure_months"] = (
            live_enriched["tenure_months_y"]
            .fillna(
                live_enriched[
                    "tenure_months_x"]))
        live_enriched.drop(
            columns=[
                "tenure_months_x",
                "tenure_months_y"],
            inplace=True)
        print(
            "  Fixed: tenure_months"
            " duplicate resolved")
    elif "tenure_months_x" in \
            live_enriched.columns:
        live_enriched.rename(
            columns={
                "tenure_months_x":
                    "tenure_months"},
            inplace=True)

    del abt_raw, abt_latest, abt_for_merge
    gc.collect()

else:
    print(
        "  abt_live not found —"
        " using scores only")
    live_enriched = live_active.copy()

# Assign priority tiers WITH guardrail
live_enriched = assign_priority_tiers(
    live_enriched,
    tenure_col="tenure_months",
    apply_nj_guardrail=True
)

live_enriched["priority_flag"] = np.where(
    live_enriched["risk_band"] == "High",
    "HIGH",
    np.where(
        live_enriched["risk_band"] ==
        "Medium",
        "MONITOR", "LOW"
    )
)
live_enriched["retention_flag"]  = ""
live_enriched["model_override"]  = False
live_enriched["override_reason"] = ""

# Print final distribution
print(
    f"\n  Active HC Risk Distribution:")
print(
    f"  Employees in HRBP list :"
    f" {len(live_enriched):,}")
for band in ["High", "Medium", "Low"]:
    n = int(
        (live_enriched["risk_band"]
         == band).sum())
    pct = float(
        n / max(len(live_enriched), 1)
        * 100)
    print(
        f"    {band:<8}: {n:>6,}"
        f" ({pct:.1f}%)")

print(f"\n  Priority Tier Distribution:")
for tier in [
    "P1 - Urgent", "P2 - High",
    "P3 - Monitor", "P4 - Watch List"
]:
    n = int(
        (live_enriched["priority_tier"]
         == tier).sum())
    print(f"    {tier:<18}: {n:>6,}")

# New joiner breakdown
if "is_new_joiner" in \
        live_enriched.columns:
    print(f"\n  New Joiner Breakdown:")
    nj_df = live_enriched[
        live_enriched[
            "is_new_joiner"] == 1]
    print(
        f"  Total new joiners :"
        f" {len(nj_df):,}")
    for tier in [
        "P1 - Urgent", "P2 - High",
        "P3 - Monitor",
        "P4 - Watch List"
    ]:
        n = int(
            (nj_df["priority_tier"]
             == tier).sum())
        print(
            f"    {tier:<18}: {n:>5,}")
    nj_p1p2 = int(
        nj_df["priority_tier"]
        .isin([
            "P1 - Urgent",
            "P2 - High"])
        .sum())
    status = (
        "PASSED" if nj_p1p2 == 0
        else "FAIL")
    print(
        f"  P1+P2 for new joiners :"
        f" {nj_p1p2}  [{status}]")

# Save
live_enriched.to_parquet(
    os.path.join(
        PATH_OUTPUT, "scores",
        "live_hrbp_priority.parquet"),
    index=False)
live_enriched.to_csv(
    os.path.join(
        PATH_OUTPUT, "scores",
        "live_hrbp_priority.csv"),
    index=False)

print(f"\n  live_hrbp_priority saved ✅")
print(
    f"  Employees :"
    f" {len(live_enriched):,}")
print(f"  Columns:")
for c in sorted(live_enriched.columns):
    print(f"    {c}")


# ============================================================
# SECTION 9 — SHAP (5k TEST SAMPLE)
# ============================================================
print("\n" + "=" * 65)
print("SECTION 9 — SHAP (5k TEST SAMPLE)")
print("=" * 65)

np.random.seed(RANDOM_STATE)
n_shap   = min(5000, len(X_test))
shap_idx = np.random.choice(
    len(X_test), n_shap, replace=False)
X_shap   = X_test[shap_idx]

print(
    f"  Computing SHAP on"
    f" {n_shap:,} test samples...")

if hasattr(champ_model, "get_booster") \
        or hasattr(
            champ_model,
            "get_feature_importance"):
    explainer = shap.TreeExplainer(
        champ_model)
    shap_values = explainer.shap_values(
        X_shap)
else:
    explainer = shap.LinearExplainer(
        champ_model, X_tr_sc[:2000])
    shap_values = explainer.shap_values(
        scaler.transform(X_shap))

plt.figure(figsize=(12, 9))
shap.summary_plot(
    shap_values, X_shap,
    feature_names=feature_names,
    max_display=20, show=False)
plt.title(
    f"SHAP — {champion_m['name']} |"
    f" AUC-PR:"
    f"{champion_m['auc_pr']:.4f}",
    fontsize=11, pad=12)
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "shap",
        "shap_global_summary.png"),
    dpi=150, bbox_inches="tight")
plt.close()

mean_shap = np.abs(
    shap_values).mean(axis=0)
feat_imp  = pd.DataFrame({
    "feature":       feature_names,
    "mean_abs_shap": mean_shap,
}).sort_values(
    "mean_abs_shap", ascending=False
).reset_index(drop=True)
feat_imp["rank"] = feat_imp.index + 1

fig, ax = plt.subplots(figsize=(11, 8))
top20 = feat_imp.head(20)
ax.barh(
    top20["feature"][::-1],
    top20["mean_abs_shap"][::-1],
    color=C_NAVY)
ax.set(
    xlabel="Mean |SHAP|",
    title="Top 20 Features")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "shap",
        "shap_feature_importance_bar.png"),
    dpi=150, bbox_inches="tight")
plt.close()

top5 = feat_imp[
    "feature"].head(5).tolist()
fig, axes = plt.subplots(
    1, 5, figsize=(22, 5))
for i, feat in enumerate(top5):
    fi = feature_names.index(feat)
    axes[i].scatter(
        X_shap[:, fi],
        shap_values[:, fi],
        c=shap_values[:, fi],
        cmap="RdBu_r",
        alpha=0.4, s=8)
    axes[i].axhline(
        y=0, color="k",
        lw=0.8, linestyle="--")
    axes[i].set(
        xlabel=feat[:25],
        ylabel="SHAP" if i == 0 else "",
        title=f"Dep: {feat[:20]}")
    axes[i].grid(alpha=0.3)
plt.suptitle(
    "SHAP Dependence — Top 5",
    fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "shap",
        "shap_dependence_top5.png"),
    dpi=150, bbox_inches="tight")
plt.close()
print(
    "  Plot saved:"
    " shap_dependence_top5.png")

feat_imp.to_csv(
    os.path.join(
        PATH_OUTPUT, "shap",
        "feature_importance.csv"),
    index=False)

test_meta_shap = pd.read_parquet(
    os.path.join(
        PATH_DATA, "abt_test.parquet"),
    columns=[
        "Employee_ID", "Month_Date"]
).iloc[shap_idx].reset_index(drop=True)

shap_rows = []
for i in range(n_shap):
    sv   = shap_values[i]
    top3 = np.argsort(
        np.abs(sv))[::-1][:3]
    score = float(
        champ_test_prob[shap_idx[i]])
    band  = assign_risk_bands(
        np.array([score]))[0]
    shap_rows.append({
        "Employee_ID":
            test_meta_shap.iloc[i][
                "Employee_ID"],
        "attrition_score":
            round(score, 4),
        "risk_band": band,
        "driver_1_feature":
            feature_names[top3[0]],
        "driver_1_direction": (
            "increases"
            if sv[top3[0]] > 0
            else "reduces"),
        "driver_1_shap":
            round(float(sv[top3[0]]),
                  4),
        "driver_2_feature":
            feature_names[top3[1]],
        "driver_2_direction": (
            "increases"
            if sv[top3[1]] > 0
            else "reduces"),
        "driver_2_shap":
            round(float(sv[top3[1]]),
                  4),
        "driver_3_feature":
            feature_names[top3[2]],
        "driver_3_direction": (
            "increases"
            if sv[top3[2]] > 0
            else "reduces"),
        "driver_3_shap":
            round(float(sv[top3[2]]),
                  4),
    })

df_shap_out = pd.DataFrame(shap_rows)
df_shap_out.to_parquet(
    os.path.join(
        PATH_OUTPUT, "shap",
        "employee_shap_drivers.parquet"),
    index=False)
print(
    f"  SHAP drivers saved:"
    f" {len(df_shap_out):,} employees")

print(f"\n  Top 20 features:")
print(
    f"  {'Rank':<5} {'Feature':<45}"
    f" Mean|SHAP|")
print("  " + "-" * 58)
for _, r in feat_imp.head(20).iterrows():
    print(
        f"  {int(r['rank']):<5}"
        f" {r['feature']:<45}"
        f" {r['mean_abs_shap']:.5f}")


# ============================================================
# SECTION 9B — SHAP FULL LIVE POPULATION
# Now runs on ~25k rows (latest month only)
# instead of ~99k rows — 4x faster and
# reflects current employee state ✅
# ============================================================
print("\n" + "=" * 65)
print("SECTION 9B — SHAP FULL LIVE"
      " (LATEST SNAPSHOT)")
print("=" * 65)
print(
    f"  Population: {len(X_live):,}"
    f" employees (Feb 2026 only)")
print(
    f"  Previous run: 99,097 rows"
    f" (all historical snapshots)")
print(
    f"  Improvement: 1 row/employee,"
    f" current state, ~4x faster")

if not RECOMPUTE_SHAP:
    print("  Loading from cache...")
    sc = load_cache(SHAP_FULL_CACHE)
    shap_live_vals  = sc["shap_values"]
    live_emp_ids    = sc["live_emp_ids"]
    feat_imp_full   = sc["feat_imp_full"]
    df_drivers_full = sc["df_drivers_full"]
    print(
        f"  Loaded"
        f" {len(df_drivers_full):,}"
        f" employees")

else:
    print(
        f"  Computing SHAP for"
        f" {len(X_live):,} employees...")
    print(
        f"  Batches of 5,000"
        f" (~4 min total)")

    if hasattr(
            champ_model, "get_booster") \
            or hasattr(
                champ_model,
                "get_feature_importance"):
        live_expl = shap.TreeExplainer(
            champ_model)
    else:
        live_expl = shap.LinearExplainer(
            champ_model, X_tr_sc[:2000])

    BATCH   = 5000
    n_live  = len(X_live)
    n_bat   = int(np.ceil(n_live / BATCH))
    sv_list = []

    for bi in range(n_bat):
        s  = bi * BATCH
        e  = min(s + BATCH, n_live)
        Xb = X_live[s:e]
        if hasattr(
                champ_model,
                "get_booster") or \
                hasattr(
                    champ_model,
                    "get_feature_importance"):
            sv = live_expl.shap_values(Xb)
        else:
            sv = live_expl.shap_values(
                scaler.transform(Xb))
        sv_list.append(sv)
        print(
            f"    Batch {bi+1}/{n_bat}"
            f" rows {s:,}-{e:,}")

    shap_live_vals = np.vstack(sv_list)
    del sv_list
    gc.collect()
    print(
        f"  shape:"
        f" {shap_live_vals.shape}")

    # live_emp_ids already 1/employee
    # from preprocess latest-month filter
    live_emp_ids = (
        live_ids["Employee_ID"].values)

    msf = np.abs(
        shap_live_vals).mean(axis=0)
    feat_imp_full = pd.DataFrame({
        "feature":       feature_names,
        "mean_abs_shap": msf,
    }).sort_values(
        "mean_abs_shap", ascending=False
    ).reset_index(drop=True)
    feat_imp_full["rank"] = (
        feat_imp_full.index + 1)

    print(
        "  Extracting top-3 drivers...")
    dl = []
    for i in range(len(X_live)):
        sv   = shap_live_vals[i]
        top3 = np.argsort(
            np.abs(sv))[::-1][:3]
        score = float(champ_live_prob[i])
        band  = assign_risk_bands(
            np.array([score]))[0]
        dl.append({
            "Employee_ID":
                live_emp_ids[i],
            "attrition_score":
                round(score, 4),
            "risk_band": band,
            "driver_1_feature":
                feature_names[top3[0]],
            "driver_1_direction": (
                "increases"
                if sv[top3[0]] > 0
                else "reduces"),
            "driver_1_shap": round(
                float(sv[top3[0]]), 4),
            "driver_2_feature":
                feature_names[top3[1]],
            "driver_2_direction": (
                "increases"
                if sv[top3[1]] > 0
                else "reduces"),
            "driver_2_shap": round(
                float(sv[top3[1]]), 4),
            "driver_3_feature":
                feature_names[top3[2]],
            "driver_3_direction": (
                "increases"
                if sv[top3[2]] > 0
                else "reduces"),
            "driver_3_shap": round(
                float(sv[top3[2]]), 4),
        })
    df_drivers_full = pd.DataFrame(dl)

    save_cache(SHAP_FULL_CACHE, {
        "shap_values"    : shap_live_vals,
        "live_emp_ids"   : live_emp_ids,
        "feat_imp_full"  : feat_imp_full,
        "df_drivers_full": df_drivers_full,
        "data_hash"      : get_data_hash(),
    })

df_drivers_full.to_parquet(
    os.path.join(
        PATH_OUTPUT, "shap",
        "employee_shap_drivers_full"
        ".parquet"),
    index=False)
feat_imp_full.to_csv(
    os.path.join(
        PATH_OUTPUT, "shap",
        "feature_importance_full.csv"),
    index=False)

print(
    f"\n  Full SHAP saved —"
    f" {len(df_drivers_full):,}"
    f" employees (latest snapshot)")

print(f"\n  Top 10 (full live):")
print(
    f"  {'Rank':<5} {'Feature':<45}"
    f" Mean|SHAP|")
print("  " + "-" * 58)
for _, r in \
        feat_imp_full.head(10).iterrows():
    print(
        f"  {int(r['rank']):<5}"
        f" {r['feature']:<45}"
        f" {r['mean_abs_shap']:.5f}")

print(f"\n  Rank drift (test vs live):")
print(
    f"  {'Feature':<40}"
    f" {'Test':>6}"
    f" {'Live':>6}"
    f" {'Drift':>7}")
print("  " + "-" * 62)
for _, r in feat_imp.head(15).iterrows():
    feat = r["feature"]
    tr   = int(r["rank"])
    lr_  = feat_imp_full[
        feat_imp_full["feature"] == feat]
    if len(lr_) > 0:
        lrk  = int(
            lr_["rank"].values[0])
        d    = lrk - tr
        ds   = (
            f"+{d}" if d > 0 else str(d))
        flag = (
            " WARNING"
            if abs(d) > 5 else "")
        print(
            f"  {feat:<40} {tr:>6}"
            f" {lrk:>6} {ds:>7}{flag}")


# ============================================================
# SECTION 10 — BIAS AUDIT
# ============================================================
print("\n" + "=" * 65)
print("SECTION 10 — BIAS AUDIT")
print("=" * 65)

bias_raw = pd.read_parquet(
    os.path.join(
        PATH_DATA, "abt_test.parquet"),
    columns=[
        "Employee_ID", "Month_Date",
        "gender_encoded", "Region",
        "mgmt_ordinal",
        "attrition_label",
        "tenure_months",
        "is_new_joiner"])
bias_raw["attrition_score"] = (
    champ_test_prob)
bias_raw["risk_band"] = assign_risk_bands(
    champ_test_prob)
bias_raw["predicted_high"] = (
    bias_raw["risk_band"] == "High"
).astype(int)
bias_raw["gender_label"] = (
    bias_raw["gender_encoded"].map(
        {0: "Male", 1: "Female",
         2: "Prefer Not",
         -1: "Unknown"}))

# Exclude new joiners from bias audit
# Their distribution is intentionally
# different due to guardrail — including
# them would inflate false bias flags
bias_reg = bias_raw[
    bias_raw["is_new_joiner"] == 0
].copy()
bias_nj  = bias_raw[
    bias_raw["is_new_joiner"] == 1
].copy()

print(
    f"\n  Bias audit on"
    f" non-new-joiners:")
print(
    f"  (tenure >= 18m,"
    f" {len(bias_reg):,} rows)")
print(
    f"  New joiners excluded:"
    f" {len(bias_nj):,}")

bias_flags = []

def bias_table(df, grp_col, label=""):
    g = df.groupby(grp_col).agg(
        n      =(
            "Employee_ID", "count"),
        leavers=(
            "attrition_label", "sum"),
        high   =(
            "predicted_high",  "sum"),
        avg_sc =(
            "attrition_score", "mean"),
    ).reset_index()
    g["leaver_rate_pct"] = (
        g["leavers"] /
        g["n"] * 100).round(2)
    g["high_risk_rate_pct"] = (
        g["high"] /
        g["n"] * 100).round(2)
    print(f"\n  {label}:")
    print(
        g.rename(
            columns={grp_col: "Group"}
        ).to_string(index=False))
    for _, r in g.iterrows():
        if (r["leaver_rate_pct"] > 0 and
                r["high_risk_rate_pct"] >
                2 * r["leaver_rate_pct"]):
            bias_flags.append(
                f"  {label} —"
                f" {r[grp_col]}:"
                f" high"
                f" {r['high_risk_rate_pct']:.1f}%"
                f" vs actual"
                f" {r['leaver_rate_pct']:.1f}%")
    return g

grp_gender = bias_table(
    bias_reg,
    "gender_label", "Gender")
grp_region = bias_table(
    bias_reg, "Region", "Region")
grp_mgmt   = bias_table(
    bias_reg,
    "mgmt_ordinal", "Mgmt Level")

def bias_bar(grp, gc, label, fname):
    fig, ax = plt.subplots(
        figsize=(
            max(8, len(grp) * 1.5), 5))
    x = np.arange(len(grp))
    w = 0.35
    b1 = ax.bar(
        x - w/2,
        grp["leaver_rate_pct"], w,
        color=C_NAVY,
        label="Actual Leaver %")
    b2 = ax.bar(
        x + w/2,
        grp["high_risk_rate_pct"], w,
        color=C_RED,
        label="High Risk %")
    ax.set(
        xticks=x,
        xticklabels=[
            str(v)[:18]
            for v in grp[gc]],
        ylabel="%",
        title=f"Bias: {label}")
    ax.legend()
    ax.grid(alpha=0.3, axis="y")
    for b in [b1, b2]:
        for bar in b:
            ax.text(
                bar.get_x() +
                bar.get_width() / 2,
                bar.get_height() + 0.05,
                f"{bar.get_height():.1f}%",
                ha="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            PATH_OUTPUT, "plots",
            fname),
        dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Plot saved: {fname}")

bias_bar(
    grp_gender, "gender_label",
    "Gender", "bias_gender.png")
bias_bar(
    grp_region, "Region",
    "Region", "bias_region.png")
bias_bar(
    grp_mgmt, "mgmt_ordinal",
    "Mgmt Level", "bias_mgmt.png")

pd.concat([
    grp_gender.assign(
        dimension="Gender"),
    grp_region.assign(
        dimension="Region"),
    grp_mgmt.assign(
        dimension="Mgmt Level")
]).to_csv(
    os.path.join(
        PATH_OUTPUT, "scores",
        "bias_audit.csv"),
    index=False)
print("  bias_audit.csv saved")

if bias_flags:
    print("\n  Bias flags:")
    for f in bias_flags:
        print(f"    {f}")
else:
    print("\n  No bias flags raised ✅")


# ============================================================
# SECTION 11 — TENURE COHORT ANALYSIS
# ============================================================
print("\n" + "=" * 65)
print("SECTION 11 — TENURE COHORT ANALYSIS")
print("=" * 65)

tenure_raw = pd.read_parquet(
    os.path.join(
        PATH_DATA, "abt_test.parquet"),
    columns=[
        "Employee_ID",
        "tenure_months",
        "attrition_label",
        "is_new_joiner"])
tenure_raw["attrition_score"] = (
    champ_test_prob)
tenure_raw["predicted_high"]  = (
    assign_risk_bands(champ_test_prob)
    == "High").astype(int)

bins   = [0, 12, 24, 36, 48, 60, 9999]
labels = [
    "0-12m", "12-24m", "24-36m",
    "36-48m", "48-60m", "60m+"]
tenure_raw["tenure_band"] = pd.cut(
    tenure_raw["tenure_months"],
    bins=bins, labels=labels,
    right=False)

cohort = tenure_raw.groupby(
    "tenure_band",
    observed=True).agg(
    n          =(
        "Employee_ID",    "count"),
    leavers    =(
        "attrition_label","sum"),
    high_risk  =(
        "predicted_high", "sum"),
    avg_score  =(
        "attrition_score","mean"),
    new_joiners=(
        "is_new_joiner",  "sum"),
).reset_index()
cohort["actual_leaver_rate_pct"] = (
    cohort["leavers"] /
    cohort["n"] * 100).round(2)
cohort["high_risk_rate_pct"] = (
    cohort["high_risk"] /
    cohort["n"] * 100).round(2)
cohort["new_joiner_pct"] = (
    cohort["new_joiners"] /
    cohort["n"] * 100).round(1)

print("\n  Tenure cohort analysis:")
print(cohort.to_string(index=False))

print(
    f"\n  Cohorts where new joiner"
    f" guardrail applies"
    f" (tenure < {NEW_JOINER_TRIAGE_MONTHS}m):")
for _, r in cohort.iterrows():
    if r["new_joiners"] > 0:
        print(
            f"    {str(r['tenure_band']):<8}"
            f" : {int(r['new_joiners']):>6,}"
            f" new joiners"
            f" ({r['new_joiner_pct']:.1f}%)"
            f" — P1/P2 capped to P3")

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
x = np.arange(len(cohort))
w = 0.35
ax1.bar(
    x - w/2,
    cohort["actual_leaver_rate_pct"],
    w, color=C_NAVY, alpha=0.85,
    label="Actual Leaver %")
ax1.bar(
    x + w/2,
    cohort["high_risk_rate_pct"],
    w, color=C_AMBER, alpha=0.85,
    label="High Risk %")
ax2.plot(
    x, cohort["avg_score"] * 100,
    color=C_RED, marker="o", lw=2,
    label="Avg Score x100")
ax1.set(
    xticks=x, xticklabels=labels,
    xlabel="Tenure Band",
    ylabel="%",
    title="Risk by Tenure Cohort"
          " (guardrail applied)")
ax2.set_ylabel(
    "Avg Score x100", color=C_RED)
ax2.tick_params(
    axis="y", labelcolor=C_RED)
l1, lb1 = (
    ax1.get_legend_handles_labels())
l2, lb2 = (
    ax2.get_legend_handles_labels())
ax1.legend(
    l1 + l2, lb1 + lb2,
    loc="upper right")
ax1.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(
    os.path.join(
        PATH_OUTPUT, "plots",
        "tenure_cohort_analysis.png"),
    dpi=150, bbox_inches="tight")
plt.close()
print(
    "  Plot saved:"
    " tenure_cohort_analysis.png")


# ============================================================
# SECTION 12 — FINAL SUMMARY
# ============================================================
print("\n" + "=" * 65)
print("SECTION 12 — FINAL SUMMARY")
print("=" * 65)

print(
    f"\n  Champion  :"
    f" {champion_m['name']}")
print(
    f"  AUC-ROC   :"
    f" {champion_m['auc_roc']:.4f}")
print(
    f"  AUC-PR    :"
    f" {champion_m['auc_pr']:.4f}")
print(
    f"  Recall@10%:"
    f" {champion_m['recall_decile']*100:.1f}%")
print(
    f"  Best F1   :"
    f" {champion_m['best_f1']:.4f}")

print(f"\n  Thresholds (absolute):")
print(
    f"    High   >= {HIGH_THRESH}"
    f"  (~18% of active HC)")
print(
    f"    Medium >= {MED_THRESH}"
    f"  (~13% of active HC)")
print(
    f"    H+M total ~31%"
    f" — operationally feasible")

print(f"\n  Priority tiers:")
print(
    f"    P1 Urgent  :"
    f" Top {PRIORITY_1_N:,} by score")
print(
    f"    P2 High    :"
    f" Next {PRIORITY_2_N:,} by score")
print(
    f"    P3 Monitor : Rest of H+M")
print(
    f"    P4 Watch   : Low band")

print(f"\n  New joiner guardrail:")
print(
    f"    Tenure < "
    f"{NEW_JOINER_TRIAGE_MONTHS}m"
    f" -> capped at P3")
print(
    f"    Aligned to ABT threshold ✅")
print(
    f"    Triage logic only"
    f" — not a model change")

print(f"\n  SHAP population:")
print(
    f"    {len(X_live):,} employees"
    f" (Feb 2026 latest snapshot only)")
print(
    f"    1 row per employee ✅")
print(
    f"    Current state features ✅")

print(f"\n  TEST risk band capture:")
for band in ["High", "Medium", "Low"]:
    bd  = test_meta[
        test_meta["risk_band"] == band]
    nlv = int(bd["actual_leaver"].sum())
    pct = float(
        nlv / max(y_test.sum(), 1) * 100)
    print(
        f"    {band:<8}: {len(bd):>7,} |"
        f" {nlv:>5,} leavers"
        f" ({pct:.1f}% of all)")

print(f"\n  LIVE active HC (post-filter):")
for band in ["High", "Medium", "Low"]:
    n = int(
        (live_enriched["risk_band"]
         == band).sum())
    pct = float(
        n / max(
            len(live_enriched), 1) * 100)
    print(
        f"    {band:<8}: {n:>6,}"
        f" ({pct:.1f}%)")

print(f"\n  Priority tier distribution:")
for tier in [
    "P1 - Urgent", "P2 - High",
    "P3 - Monitor", "P4 - Watch List"
]:
    n = int(
        (live_enriched["priority_tier"]
         == tier).sum())
    print(f"    {tier:<18}: {n:>6,}")

if "is_new_joiner" in \
        live_enriched.columns:
    nj_p1p2 = int(
        live_enriched[
            live_enriched[
                "is_new_joiner"] == 1
        ]["priority_tier"]
        .isin([
            "P1 - Urgent",
            "P2 - High"])
        .sum())
    status = (
        "PASSED" if nj_p1p2 == 0
        else "FAIL")
    print(
        f"\n  New joiner P1+P2 check:"
        f" {nj_p1p2}  [{status}]")

print(f"\n  Top 10 SHAP (test sample):")
for _, r in feat_imp.head(10).iterrows():
    print(
        f"    {int(r['rank']):>2}."
        f" {r['feature']:<42}"
        f" {r['mean_abs_shap']:.5f}")

print(f"\n  Top 10 SHAP (full live):")
for _, r in \
        feat_imp_full.head(10).iterrows():
    print(
        f"    {int(r['rank']):>2}."
        f" {r['feature']:<42}"
        f" {r['mean_abs_shap']:.5f}")

if bias_flags:
    print(
        f"\n  Bias flags:"
        f" {len(bias_flags)}")
    for f in bias_flags:
        print(f"    {f}")
else:
    print("\n  Bias audit: 0 flags ✅")

print(
    f"\n  SHAP:"
    f" {'cached' if not RECOMPUTE_SHAP else 'freshly computed'}"
    f" — {len(df_drivers_full):,}"
    f" live employees"
    f" (latest snapshot only)")

print(f"\n  Output: {PATH_OUTPUT}")
print(
    f"  plots/ : roc/pr/learning/"
    f"comparison/score_dist/bias/tenure")
print(
    f"  shap/  : summary/bar/dependence"
    f" importance*.csv drivers*.parquet")
print(
    f"  scores/: test_scores"
    f" live_scores"
    f" live_hrbp_priority bias_audit")
print(
    f"  models/: xgb_champion.pkl"
    f" catboost_challenger.cbm")
print(
    f"  cache/ : shap_full_population"
    f" label_encoders imputer"
    f" feature_names")

print("\n" + "=" * 65)
print("MODEL TRAINING COMPLETE ✅")
print("=" * 65)